In [2]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import faiss
import wandb
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sys.path.append(os.path.abspath('..'))
from src.utils import map_at_3, load_config, set_seed

warnings.filterwarnings('ignore')
load_dotenv('../.env')
set_seed(42)

OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
TRAIN_PATH  = '../data/raw/train.csv'

train = pd.read_csv(TRAIN_PATH)
print(f'Shape: {train.shape}')

/Users/risshabsrinivas/Documents/IITM/DL_GENAI_PROJECT/dl-genai-project-26-t2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Shape: (2000, 8)


In [3]:
# ── Knowledge Base + FAISS Index ──────────────────────────────────────────────
print('Building knowledge base...')
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print('Loading embedding model and creating FAISS index...')
bi_encoder    = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = bi_encoder.encode(kb, show_progress_bar=True)
kb_embeddings = np.array(kb_embeddings).astype('float32')

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print(f'Knowledge base size : {len(kb)}')
print(f'FAISS index size    : {index.ntotal}')
print('Knowledge base successfully created.')

Building knowledge base...
Loading embedding model and creating FAISS index...


Batches: 100%|██████████| 63/63 [00:02<00:00, 31.31it/s]

Knowledge base size : 2000
FAISS index size    : 2000
Knowledge base successfully created.


In [4]:
# ── Shared objects used across multiple questions ─────────────────────────────
zs          = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

row_150     = train.iloc[150]
prompt_150  = str(row_150['prompt'])
labels_150  = [str(row_150[c]) for c in OPTION_COLS]
ans_150     = str(row_150[row_150['answer']])

print(f'Row 150 prompt  : {prompt_150[:80]}...')
print(f'Row 150 answer  : {row_150["answer"]} → {ans_150[:60]}...')

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8319.36it/s]


Row 150 prompt  : Select the most accurate option: What is the butterfly effect, as defined by Lor...
Row 150 answer  : C → The butterfly effect is the phenomenon that a small change i...


In [5]:
# ── Shared objects used across multiple questions ─────────────────────────────
zs          = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

row_150     = train.iloc[150]
prompt_150  = str(row_150['prompt'])
labels_150  = [str(row_150[c]) for c in OPTION_COLS]
ans_150     = str(row_150[row_150['answer']])

print(f'Row 150 prompt  : {prompt_150[:80]}...')
print(f'Row 150 answer  : {row_150["answer"]} → {ans_150[:60]}...')

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 20281.00it/s]


Row 150 prompt  : Select the most accurate option: What is the butterfly effect, as defined by Lor...
Row 150 answer  : C → The butterfly effect is the phenomenon that a small change i...


In [6]:
prompt_150_emb = bi_encoder.encode([prompt_150], show_progress_bar=False)
prompt_150_emb = np.array(prompt_150_emb).astype('float32')

k = 10
distances, retrieved_indices = index.search(prompt_150_emb, k)
retrieved_indices = retrieved_indices[0].tolist()

print(f'Top-{k} retrieved KB indices: {retrieved_indices}')
print(f'True document is at KB index: 150')

if 150 in retrieved_indices:
    q2_answer = retrieved_indices.index(150) + 1   # 1-indexed rank
else:
    q2_answer = 'Not in top-10'

print(f'(rank of true doc): {q2_answer}')

Top-10 retrieved KB indices: [663, 1701, 1269, 1532, 576, 847, 1693, 1906, 168, 150]
True document is at KB index: 150
(rank of true doc): 10


In [7]:
docs_10 = [kb[i] for i in retrieved_indices]
pairs   = [[prompt_150, doc] for doc in docs_10]

ce_scores        = cross_encoder.predict(pairs)
reranked_indices = np.argsort(ce_scores)[::-1].tolist()   # descending
reranked_kb_ids  = [retrieved_indices[i] for i in reranked_indices]

print(f'CE scores           : {[round(s, 4) for s in ce_scores]}')
print(f'Reranked KB indices : {reranked_kb_ids}')

if 150 in reranked_kb_ids:
    q3_answer = reranked_kb_ids.index(150) + 1
else:
    q3_answer = 'Not in top-10'

print(f'(cross-encoder rank of true doc): {q3_answer}')

CE scores           : [np.float32(4.6602), np.float32(4.6602), np.float32(4.7375), np.float32(4.7375), np.float32(4.687), np.float32(4.7526), np.float32(4.7526), np.float32(4.7526), np.float32(4.7072), np.float32(4.7585)]
Reranked KB indices : [150, 1906, 1693, 847, 1532, 1269, 168, 576, 1701, 663]
(cross-encoder rank of true doc): 1


In [8]:
row_42     = train.iloc[42]
prompt_42  = str(row_42['prompt'])

emb_42     = bi_encoder.encode([prompt_42], show_progress_bar=False)
emb_42     = np.array(emb_42).astype('float32')
_, idx_42  = index.search(emb_42, 5)
docs_42    = [kb[i] for i in idx_42[0].tolist()]

concatenated = ' '.join(docs_42)
rag_string   = f'Context: {concatenated} Question: {prompt_42}'

tokens   = bert_tokenizer(rag_string, truncation=False)
q4_answer = len(tokens['input_ids'])

print(f'RAG string (first 150 chars): {rag_string[:150]}...')
print(f'(total tokens): {q4_answer}')

RAG string (first 150 chars): Context: Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion ...
(total tokens): 216


In [10]:
true_doc_150 = kb[150]
rag_str_150  = f'Context: {true_doc_150} Question: {prompt_150}'

zs_rag_150       = zs(rag_str_150, candidate_labels=labels_150)
label_to_score_rag = dict(zip(zs_rag_150['labels'], zs_rag_150['scores']))
q5_answer          = round(label_to_score_rag[ans_150], 3)

print(f'RAG string (first 120 chars): {rag_str_150[:120]}...')
print(f'Labels : {zs_rag_150["labels"]}')
print(f'Scores : {[round(s, 3) for s in zs_rag_150["scores"]]}')
print(f'(RAG ground-truth prob): {q5_answer}')


RAG string (first 120 chars): Context: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework c...
Labels : ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states tha

In [11]:
adversarial_doc = kb[999]
adv_rag_str     = f'Context: {adversarial_doc} Question: {prompt_150}'

zs_adv               = zs(adv_rag_str, candidate_labels=labels_150)
label_to_score_adv   = dict(zip(zs_adv['labels'], zs_adv['scores']))
q6_answer            = round(label_to_score_adv[ans_150], 3)

print(f'Adversarial doc (KB 999): {adversarial_doc[:80]}...')
print(f'Scores : {[round(s, 3) for s in zs_adv["scores"]]}')
print(f'(adversarial RAG ground-truth prob): {q6_answer}')
print(f'\nComparison:')

print(f'  True RAG     (Q5): {q5_answer}')
print(f'  Adversarial  (Q6): {q6_answer}')

Adversarial doc (KB 999): A thought experiment in which a demon guards a microscopic trapdoor in a wall se...
Scores : [0.529, 0.425, 0.02, 0.019, 0.007]
(adversarial RAG ground-truth prob): 0.529

Comparison:
  True RAG     (Q5): 0.989
  Adversarial  (Q6): 0.529
